In [1]:
# Importações básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Bibliotecas especializadas
# import missingno as msno
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
# import category_encoders as ce
import math
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

print(" Bibliotecas importadas!")

 Bibliotecas importadas!


Carregando as colunas retirando: 
rest_index /
attr_index_norm /
rest_index_norm /
room_shared /
room_private / 
lat /
lng /
Unnamed 

In [2]:
from pathlib import Path
import pandas as pd
import os

# Caminho da pasta onde estão os CSVs
path = Path('../../data/Airbnb Prices in European Cities/raw')

list_of_dfs = []

for file in os.listdir(path):
    if file.endswith(".csv"):
        
        file_path = path / file

        # ✅ Lê o CSV
        df_temp = pd.read_csv(file_path)

        # ✅ Remove colunas indesejadas
        df_temp = df_temp.drop(
            columns=[
                "rest_index",
                "attr_index_norm",
                "rest_index_norm",
                "room_shared",
                "room_private",
                "lat",
                "lng",
                "version https://git-lfs.github.com/spec/v1"
            ],
            errors="ignore"  # não quebra se alguma não existir
        )

        # ✅ Remove qualquer coluna "Unnamed"
        df_temp = df_temp.loc[:, ~df_temp.columns.str.contains("^Unnamed")]

        # ✅ Adiciona coluna indicando de qual arquivo veio
        df_temp["source_file"] = file.replace(".csv", "")

        list_of_dfs.append(df_temp)

# ✅ Junta tudo em um único DataFrame
df = pd.concat(list_of_dfs, ignore_index=True)

print("\n--------------------------------------------")
print("📊 Todos os dados foram combinados em `df`")
print(f"Total de linhas: {len(df)}")
print(f"Colunas finais: {df.columns.tolist()}")



--------------------------------------------
📊 Todos os dados foram combinados em `df`
Total de linhas: 50425
Colunas finais: ['source_file', 'realSum', 'room_type', 'person_capacity', 'host_is_superhost', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index']


Colocando a feature Cidade e Tipo_dia

In [3]:
from pathlib import Path
import pandas as pd
import os

# Caminho da pasta onde estão os CSVs
path = Path('../../data/Airbnb Prices in European Cities/raw')

list_of_dfs = []

for file in os.listdir(path):
    if file.endswith(".csv"):
        
        file_path = path / file

        # ✅ Lê o CSV
        df_temp = pd.read_csv(file_path)

        # ✅ Remove qualquer coluna "Unnamed"
        df_temp = df_temp.loc[:, ~df_temp.columns.str.contains("^Unnamed")]

        # ✅ Remove qualquer coluna relacionada ao Git LFS
        df_temp = df_temp.loc[:, ~df_temp.columns.str.contains("git-lfs|version", case=False)]

        # ✅ Remove TODAS as colunas pedidas anteriormente
        df_temp = df_temp.drop(
            columns=[
                "rest_index",
                "attr_index_norm",
                "rest_index_norm",
                "room_shared",
                "room_private",
                "lat",
                "lng"
            ],
            errors="ignore"
        )

        # ✅ Nome base (ex: amsterdam_weekdays)
        source_name = file.replace(".csv", "")
        df_temp["source_file"] = source_name

        # ✅ Separar cidade e tipo_dia
        partes = source_name.split("_")
        cidade = partes[0].capitalize()
        tipo_dia = partes[1]

        df_temp["cidade"] = cidade
        df_temp["tipo_dia"] = tipo_dia

        list_of_dfs.append(df_temp)

# ✅ Junta tudo em um único DataFrame
df = pd.concat(list_of_dfs, ignore_index=True)

print("\n--------------------------------------------")
print("📊 Todos os dados foram combinados em `df`")
print(f"Total de linhas: {len(df)}")
print(f"Colunas finais: {df.columns.tolist()}")



--------------------------------------------
📊 Todos os dados foram combinados em `df`
Total de linhas: 50425
Colunas finais: ['source_file', 'cidade', 'tipo_dia', 'realSum', 'room_type', 'person_capacity', 'host_is_superhost', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index']


In [4]:
# 2. Tipos de dados
print("\n2️⃣ TIPOS DE DADOS:")
print(df.dtypes)


2️⃣ TIPOS DE DADOS:
source_file                    object
cidade                         object
tipo_dia                       object
realSum                       float64
room_type                      object
person_capacity               float64
host_is_superhost              object
multi                         float64
biz                           float64
cleanliness_rating            float64
guest_satisfaction_overall    float64
bedrooms                      float64
dist                          float64
metro_dist                    float64
attr_index                    float64
dtype: object


Feature Scaling (normalização de variaveis numéricas)

Transformar as categóricas em Numéricas

In [5]:
from sklearn.model_selection import train_test_split

# Target
y = df["realSum"]

# Features (remove target)
X = df.drop(columns=["realSum"])


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Colunas categóricas
categorical_features = X.select_dtypes(include=["object", "bool"]).columns.tolist()

# Colunas numéricas
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categóricas:", categorical_features)
print("Numéricas:", numerical_features)


Categóricas: ['source_file', 'cidade', 'tipo_dia', 'room_type', 'host_is_superhost']
Numéricas: ['person_capacity', 'multi', 'biz', 'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist', 'metro_dist', 'attr_index']


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features)
    ]
)

In [8]:
X_processed = preprocessor.fit_transform(X)

In [9]:
from sklearn.model_selection import train_test_split

# 70% treino, 30% temporário (validação + teste)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Divide os 30% em 15% validação e 15% teste
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test.shape)


Treino: (35297, 14)
Validação: (7564, 14)
Teste: (7564, 14)
